**Introduccion: Refinamiento de la funcion distancia**


El sistema base utiliza la distancia de Levenshtein, que asume que todos los errores humanos son iguales (sustituir 'a' por 's' cuesta lo mismo que 'a' por 'z').

El objetivo de esta mejora es reflejar la realidad del error humano: los usuarios fallan por cercanía física en el teclado. 

Al penalizar menos los errores "lógicos", el autocorrector priorizará candidatos que tengan sentido sobre palabras aleatorias que simplemente están a la misma distancia matemática.

In [19]:
from pathlib import Path
import re,os

ruta_libros = Path('../../libros')
files = os.listdir(ruta_libros)
words = {}
for file in files:
    with open(ruta_libros/file, 'r', encoding='UTF-8') as f:
        for linea in f:
            linea = linea.lower()
            linea = re.sub(r'[^\w\s]','', linea)

            for word in linea.split():
                words[word] = words.get(word,0) + 1
                
print(words)
print(len(words))


{'project': 1923, 'gutenbergs': 8, 'la': 44757, 'lucha': 145, 'por': 11484, 'vida': 1345, 'mala': 338, 'hierba': 89, 'by': 609, 'pío': 104, 'baroja': 124, 'this': 1084, 'ebook': 255, 'is': 517, 'for': 572, 'the': 4083, 'use': 242, 'of': 2687, 'anyone': 109, 'anywhere': 43, 'at': 330, 'no': 15636, 'cost': 65, 'and': 1562, 'with': 1047, 'almost': 43, 'restrictions': 43, 'whatsoever': 43, 'you': 1581, 'may': 330, 'copy': 263, 'it': 327, 'give': 87, 'away': 43, 'or': 1713, 'reuse': 43, 'under': 131, 'terms': 461, 'gutenberg': 659, 'license': 329, 'included': 65, 'online': 124, 'wwwgutenbergorglicense': 24, 'title': 21, 'author': 21, 'release': 21, 'date': 67, 'june': 2, '25': 18, '2013': 2, '43033': 1, 'language': 21, 'spanish': 21, 'start': 65, 'produced': 88, 'chuck': 4, 'greif': 4, 'distributed': 125, 'proofreading': 37, 'canada': 4, 'team': 37, 'httpwwwpgdpcanadanet': 4, 'book': 10, 'was': 71, 'from': 365, 'scanned': 2, 'images': 35, 'public': 112, 'domain': 90, 'material': 16, 'google

**Funciones para aplicar una edicion**

Aplicamos las 4 funciones base, insertar, borrar, reemplazar e intercambiar:

In [20]:
def insert(word: str) -> set:
    alphabet = 'abcdefghijklmnñopqrstuvwxyz'
    words_created = set()
    for i in range(len(word) + 1):
        for char in alphabet:
            new_word = word[:i] + char + word[i:]
            words_created.add(new_word)
    
    return words_created

def delete(word: str) -> set:
    words_created = set()
    for i in range(len(word)):
        new_word = word[:i] + word[i+1:]
        words_created.add(new_word)

    return words_created
    
def replace(word: str) -> set:
    alphabet = 'abcdefghijklmnñopqrstuvwxyz'
    words_created = set()
    for i in range(len(word)):
        for char in alphabet:
            new_word = word[:i] + char + word[i+1:]
            words_created.add(new_word)

    return words_created

def exchange(word: str) -> set:
    words_created = set()
    for i in range(len(word) - 1):
        new_word = word[:i] + word[i+1] + word[i] + word[i+2:]
        words_created.add(new_word)

    return words_created

**Funcion una edicion y dos ediciones**

Debemos recibir una palabra y aplicar las 4 operaciones basicas y devolver las palabras unicas a distancia uno o distancia dos:

In [21]:
def una_edicion(word:str, intercambiar: bool=False) ->set:
    set1 = insert(word)
    set2 = delete(word)
    set3 = replace(word)
    if not intercambiar:
        words_one_edition = set1.union(set2, set3)
    else:
        set4 = exchange(word)
        words_one_edition = set1.union(set2, set3, set4)

    return words_one_edition


def dos_ediciones(word:str,intercambiar:bool =False) ->set:
    palabras_ed1 = una_edicion(word, intercambiar)
    palabras_ed2 = set()
    for palabra in palabras_ed1:
        palabras_ed2 = palabras_ed2.union(una_edicion(palabra, intercambiar))
    return palabras_ed2

**Filtrado de candidatos por vocabulario**

A partir de los candidatos a una y dos ediciones, nos quedamos solo con los que existen en el corpus para evitar sugerencias imposibles.

In [22]:
def palabras_conocidas(candidatas: set, vocabulario: dict) -> set:
    return {pal for pal in candidatas if pal in vocabulario}

def candidatos(word: str, vocabulario: dict, intercambiar: bool = True) -> set:
    if word in vocabulario:
        return {word}
    c1 = palabras_conocidas(una_edicion(word, intercambiar), vocabulario)
    if c1:
        return c1
    c2 = palabras_conocidas(dos_ediciones(word, intercambiar), vocabulario)
    if c2:
        return c2
    return {word}

**Distancia de teclado + Levenshtein ponderada**

Para reemplazos, la penalización depende de qué tan cerca estén las teclas. Sustituir letras vecinas cuesta menos que sustituir letras lejanas.

In [23]:
import numpy as np

def mapa_teclado_es() -> dict:
    # Coordenadas (fila, columna) de cada letra en teclado QWERTY español
    filas = ["qwertyuiop", "asdfghjklñ", "zxcvbnm"]
    posiciones = {}
    for fila_idx, fila in enumerate(filas):
        for col_idx, letra in enumerate(fila):
            posiciones[letra] = (fila_idx, col_idx)
    return posiciones

TECLADO_POS = mapa_teclado_es() #cargamos las posiciones de las letras

def distancia_euclidea_2d(p1: np.ndarray, p2: np.ndarray) -> float:
    pos1 = np.array(p1)
    pos2 = np.array(p2)
    return np.sqrt(np.sum((pos1-pos2)**2))

def costo_reemplazo(letra_origen: str, letra_destino: str, a: float = 4.0) -> float:
    # si no cambia la letra, no hay costo
    if letra_origen == letra_destino:
        return 0.0

    pos_origen = TECLADO_POS.get(letra_origen)
    pos_destino = TECLADO_POS.get(letra_destino)

    # Si alguna letra no está en el mapa, usamos costo estándar
    if pos_origen is None or pos_destino is None:
        return 1.0

    # Distancia euclídea normalizada para no desbalancear inserción/borrado
    distancia = distancia_euclidea_2d(pos_origen, pos_destino)
    return distancia / a #hiperparametro

def distancia_levenshtein_ponderada(origen: str, destino: str) -> float:
    filas,columnas = len(origen) + 1, len(destino) + 1

    # matriz = costo mínimo para transformar palabra origen en palabra destino
    matriz = []
    for i in range(filas):
        fila=[]
        for j in range(columnas):
            fila.append(0.0)
        matriz.append(fila)

    # caso base: convertir a cadena vacía (borrados) o desde vacía (inserciones)
    for i in range(filas):
        matriz[i][0] = float(i)
    for j in range(columnas):
        matriz[0][j] = float(j)

    for i in range(1, filas):
        for j in range(1, columnas):
            costo_insertar = matriz[i][j - 1] + 1.0
            costo_borrar = matriz[i - 1][j] + 1.0
            costo_reemplazar_actual = matriz[i - 1][j - 1] + costo_reemplazo(origen[i - 1], destino[j - 1])
            matriz[i][j] = min(costo_insertar, costo_borrar, costo_reemplazar_actual)

    return matriz[filas-1][columnas-1]

**Ranking final de sugerencias**

Ordenamos por menor distancia y, en empate, por mayor frecuencia en el corpus.

In [24]:
def sugerencias(word: str, vocabulario: dict, top_k: int = 10, intercambiar: bool = True):
    cands = candidatos(word.lower(), vocabulario, intercambiar)

    ranking = []
    for cand in cands:
        d = distancia_levenshtein_ponderada(word.lower(), cand)
        f = vocabulario.get(cand, 0)
        ranking.append((cand, d, f))

    ranking.sort(key=lambda x: (x[1], -x[2], x[0]))
    return ranking[:top_k]

def corregir(word: str, vocabulario: dict, intercambiar: bool = True):
    return sugerencias(word, vocabulario, top_k=1, intercambiar=intercambiar)[0][0]

**Prueba rápida**

Ejemplos de uso con palabras mal escritas.

In [25]:
#pruebas
for word in ["hols", "escuadorn", "aventruero", "intrgia"]:
    print(word, "->", corregir(word, words))

hols -> hola
escuadorn -> ecuador
aventruero -> aventurero
intrgia -> intriga
